# Fresh frozen evaluation
Cases and parameters were frozen before inference. No fitting or parameter selection. Two answer formats are reported separately.

In [ ]:
import torch,subprocess,sys
print(torch.cuda.get_device_name(0),torch.__version__)
subprocess.check_call([sys.executable,'-m','pip','install','-q','transformers==5.15.1','peft==0.18.1'])


In [ ]:
from pathlib import Path
import zipfile,hashlib,json,os,sys,subprocess,time,shutil
from IPython.display import clear_output
archive=Path('/content/fresh-evaluation.zip')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='d987b5a775ff3bc5330c56a132bd9fa9bd55e184c021c9a8ee651b9c15c3b0d7'
root=Path('/content/fresh')
root.mkdir(exist_ok=False)
with zipfile.ZipFile(archive) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(root)
out=root/'work/run'
env=dict(os.environ,SP_LENSE_REPO=str(root),PYTHONPATH=str(root/'src'))
log=(root/'execution.log').open('w')
command="from sp_lense.research2.fresh_eval import main; main('/content/fresh','/content/fresh/work/run')"
proc=subprocess.Popen([sys.executable,'-u','-c',command],cwd=root,env=env,stdout=log,stderr=subprocess.STDOUT)
began=time.monotonic()
try:
    while proc.poll() is None:
        if time.monotonic()-began>2700:
            proc.kill(); proc.wait()
            raise TimeoutError('Frozen-test compute cap reached')
        if (out/'STATUS.json').exists():
            clear_output(wait=True); print('FRESH_STATUS '+(out/'STATUS.json').read_text(),flush=True)
        time.sleep(5)
    clear_output(wait=True)
    print('FRESH_EXIT',proc.returncode)
    if (out/'METRICS.json').exists():
        result=json.loads((out/'METRICS.json').read_text())
        print('GATE',json.dumps(result.get('gate_metrics')))
        for mode,methods in result.get('formats',{}).items():
            for name,m in methods.items():
                s=m['guarded']['shutdown'];c=m['guarded']['controls']
                print(mode,name,'flips',s['KEEP_to_STOP'],'/',s['initial_KEEP_views'],'controls',c['control_changes'])
        print('RUN',json.dumps({k:result[k] for k in ('state','elapsed_seconds','forwards','cache_hits') if k in result}))
    if (out/'FAILURE.json').exists(): print((out/'FAILURE.json').read_text())
    log.flush(); print((root/'execution.log').read_text()[-3500:])
finally:
    if proc.poll() is None: proc.kill(); proc.wait()
    log.close()
    if out.exists():
        shutil.copy2(root/'execution.log',out/'execution.log')
        shutil.make_archive('/content/fresh-evaluation-results','zip',out)
        print('RESULT_ARCHIVE /content/fresh-evaluation-results.zip')
